In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
import torch
from torch.utils.data import DataLoader
import sys
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import gc

import pyarrow as pa
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
sys.modules.pop("models", None)
sys.modules.pop("data_preparation", None)
from functionality.data_preparation import IterfeaturesDataset, SmilesDataset
from functionality.models import MolFormerXLBinaryClassifierLightning

import pyarrow.parquet as pq
import pyarrow as pa

In [ ]:
def compute_and_save_embeddings_MolFormer(model_name, smiles, labels, save_path, batch_size=1000):
    """
    Compute embeddings for a list of SMILES strings in batches and save them dynamically using Parquet

    Args:
        model_name (str): Pretrained model name
        smiles (pd.Series): Data containing SMILES strings
        labels (pd.Series): Corresponding labels
        save_path (str): Path to save computed embeddings and labels in Parquet format
        batch_size (int): Number of SMILES strings to process per batch
    """
    model = AutoModel.from_pretrained(model_name, deterministic_eval=True, trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
  
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model = torch.compile(model) 
    model.eval()

    dataset = SmilesDataset(smiles, labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=4, shuffle=False)

    # Initialize Parquet writer
    parquet_schema = pa.schema(
        [pa.field(str(i), pa.float32()) for i in range(model.config.hidden_size)] + [pa.field("label", pa.int64())]
    )
    
    with pq.ParquetWriter(save_path, parquet_schema, compression="SNAPPY") as writer:
        for batch_smiles, batch_labels in tqdm(dataloader, desc="Computing Embeddings"):
            tokens = tokenizer(batch_smiles, padding=True, truncation=True, max_length=150, return_tensors="pt")
            tokens = {k: v.to(device) for k, v in tokens.items()}
    
            with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
                outputs = model(**tokens)
    
            batch_embeddings = outputs.pooler_output.cpu().numpy()
    
            df_batch = pd.DataFrame(batch_embeddings, columns=[str(i) for i in range(batch_embeddings.shape[1])])
            df_batch["label"] = batch_labels.numpy()
            table = pa.Table.from_pandas(df_batch)

            # Write batch to Parquet file
            writer.write_table(table)

    del batch_smiles, batch_labels, tokens, outputs, batch_embeddings, df_batch
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Embeddings and labels dynamically saved to {save_path}")
    return save_path

In [ ]:
model = {
    "MolFormer": "ibm/MoLFormer-XL-both-10pct"
}
batch_size = 1000
protein_names = ['sEH', 'BRD4', 'HSA']
save_dir = '../intermediates/embeddings/'


for model_name, model_ in models.items():
    for protein_name in protein_names:
        print(f"Embeddings calculations for protein: {protein_name}")

        train_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_train.parquet')
        val_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_val.parquet')
        
        train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_train.parquet")
        val_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_val.parquet")

        compute_and_save_embeddings(model_, train_data["molecule_smiles"], train_data['binds'],  train_embeddings_path, batch_size)
        compute_and_save_embeddings(model_, val_data["molecule_smiles"], val_data['binds'], val_embeddings_path, batch_size)

In [ ]:
def train_model(model_name, protein_names, emb_path="../intermediates/embeddings"):
    for protein_name in protein_names:

        print(f"Training model for protein: {protein_name}")

        train_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_train.parquet")
        val_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_val.parquet")
        
        train_dataset = IterfeaturesDataset(train_embeddings_path)
        train_loader = DataLoader(train_dataset, batch_size=1000, num_workers=2)

        val_dataset = IterfeaturesDataset(val_embeddings_path)
        val_loader = DataLoader(val_dataset, batch_size=1000, num_workers=2)

        logger = CSVLogger("logs", name=model_name)
        early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")
        checkpoint_callback = ModelCheckpoint(
            dirpath="../checkpoints",
            filename=f"{model_name}_{protein_name}-{{epoch}}-{{val_loss:.4f}}",
            monitor="val_loss",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True,
        )

        trainer = pl.Trainer(
            max_epochs=10,
            accelerator="auto",
            devices=1,
            log_every_n_steps=2,
            callbacks=[early_stopping, checkpoint_callback],
            logger=logger,
        )


        chemberta_model = MolFormerXLBinaryClassifierLightning()
        trainer.fit(chemberta_model, train_loader, val_loader)

        os.makedirs("../intermediates/models", exist_ok=True)
        trainer.save_checkpoint(f"../intermediates/models/{model_name}_{protein_name}.ckpt")

        print(f"Completed training for protein: {protein_name}")

        del train_dataset, val_dataset, train_loader, val_loader, chemberta_model
        gc.collect()

In [ ]:
model_name = 'MolFormer'
batch_size = 1000
protein_names = ['sEH', 'BRD4', 'HSA']
train_model(model_name, protein_names)